# 响应面方法论 (RSM) — 教学笔记本

本笔记本通过一个完整的酶催化反应优化案例，逐步演示 RSM 的核心流程。

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from rsm import ExperimentDesigner, RSMModel, RSMOptimizer, RSMVisualizer
from rsm.design import Factor

## 1. 定义因子与实验设计

我们研究三个因子对酶催化产率的影响：温度、pH 和反应时间。

In [2]:
factors = [
    Factor("温度", 30, 50, "°C"),
    Factor("pH", 5, 9, ""),
    Factor("时间", 20, 60, "min"),
]
designer = ExperimentDesigner(factors)
designer.summary()

,因子,低水平,高水平,中心点,单位
0,温度,30,50,40.0,°C
1,pH,5,9,7.0,
2,时间,20,60,40.0,min


In [3]:
design_df = designer.box_behnken()
print(f"Box-Behnken 设计: {len(design_df)} 组实验")
design_df

Box-Behnken 设计: 15 组实验


,温度,pH,时间,coded_温度,coded_pH,coded_时间
0,30.0,5.0,40.0,-1.0,-1.0,0.0
1,30.0,9.0,40.0,-1.0,1.0,0.0
2,50.0,5.0,40.0,1.0,-1.0,0.0
3,50.0,9.0,40.0,1.0,1.0,0.0
4,30.0,7.0,20.0,-1.0,0.0,-1.0
5,30.0,7.0,60.0,-1.0,0.0,1.0
6,50.0,7.0,20.0,1.0,0.0,-1.0
7,50.0,7.0,60.0,1.0,0.0,1.0
8,40.0,5.0,20.0,0.0,-1.0,-1.0
9,40.0,5.0,60.0,0.0,-1.0,1.0


## 2. 模拟实验数据

在教学场景中，我们使用已知模型 + 随机噪声模拟真实实验数据：

$$y = 75 + 4.5x_1 - 2.1x_2 + 1.8x_3 - 6.2x_1^2 - 4.8x_2^2 - 3.5x_3^2 + 1.5x_1x_2 + \varepsilon$$

In [4]:
X = design_df[["coded_温度", "coded_pH", "coded_时间"]].values

np.random.seed(2024)
y = (
    75
    + 4.5 * X[:, 0] - 2.1 * X[:, 1] + 1.8 * X[:, 2]
    - 6.2 * X[:, 0]**2 - 4.8 * X[:, 1]**2 - 3.5 * X[:, 2]**2
    + 1.5 * X[:, 0] * X[:, 1] - 0.8 * X[:, 0] * X[:, 2] + 0.5 * X[:, 1] * X[:, 2]
    + np.random.normal(0, 0.6, len(X))
)

design_df["产率 (%)"] = y
design_df

,温度,pH,时间,coded_温度,coded_pH,coded_时间,产率 (%)
0,30.0,5.0,40.0,-1.0,-1.0,0.0,64.100828
1,30.0,9.0,40.0,-1.0,1.0,0.0,56.342409
2,50.0,5.0,40.0,1.0,-1.0,0.0,68.979077
3,50.0,9.0,40.0,1.0,1.0,0.0,67.809453
4,30.0,7.0,20.0,-1.0,0.0,-1.0,58.749631
5,30.0,7.0,60.0,-1.0,0.0,1.0,64.096198
6,50.0,7.0,20.0,1.0,0.0,-1.0,67.228023
7,50.0,7.0,60.0,1.0,0.0,1.0,70.004823
8,40.0,5.0,20.0,0.0,-1.0,-1.0,67.775993
9,40.0,5.0,60.0,0.0,-1.0,1.0,70.161231


## 3. 模型拟合

In [5]:
model = RSMModel(order=2)
model.fit(X, y, factor_names=["温度", "pH", "时间"])

print(f"R²  = {model.r_squared:.4f}")
print(f"Adj R² = {model.adj_r_squared:.4f}")
print(f"RMSE = {model.rmse:.4f}")
print()
model.coefficient_table()

R²  = 0.9947
Adj R² = 0.9850
RMSE = 0.6742



/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:125: RuntimeWarning: divide by zero encountered in matmul
  cov = mse * np.linalg.pinv(H.T @ H)
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:125: RuntimeWarning: overflow encountered in matmul
  cov = mse * np.linalg.pinv(H.T @ H)
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:125: RuntimeWarning: invalid value encountered in matmul
  cov = mse * np.linalg.pinv(H.T @ H)


,项,系数,标准误,t值,p值,显著
0,截距,74.883106,0.389277,192.364446,7.203682e-11,***
1,温度,3.841539,0.238383,16.115009,1.676479e-05,***
2,pH,-2.007339,0.238383,-8.420658,3.873400e-04,***
3,时间,1.931362,0.238383,8.101941,4.644736e-04,***
4,温度^2,-6.370720,0.350890,-18.155901,9.315399e-06,***
5,温度 pH,1.647199,0.337124,4.886032,4.529144e-03,**
6,温度 时间,-0.642442,0.337124,-1.905654,1.150258e-01,
7,pH^2,-4.204444,0.350890,-11.982234,7.140680e-05,***
8,pH 时间,0.639264,0.337124,1.896228,1.164255e-01,
9,时间^2,-3.492717,0.350890,-9.953885,1.747784e-04,***


In [6]:
model.anova_table()

,来源,平方和,自由度,均方,F值,p值
0,回归,423.349770,9,47.038863,103.470722,0.000038
1,残差,2.273052,5,0.45461,,
2,总计,425.622821,14,,,


## 4. 残差诊断

In [7]:
viz = RSMVisualizer(model, factor_names=["温度", "pH", "时间"])
viz.residual_diagnostics()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': '#636EFA', 'size': 8},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('JCskfr0HUED3Sr4z62dMQM0oEaCdIF' ... 'KczYS4UkCmYpzNhLhSQKZinM2EuFJA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AHCUXFOYlL8As7f1kwvevwCxt/WTC9' ... '55pFIS6b8AgMal/3+pv4D01Z5Squo/'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'marker': {'color': '#EF553B', 'size': 8},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('rID9koMZ+78cUCLquKrzv5OWW9RZ9e' ... 'Zb1Fn17T8cUCLquKrzP6yA/ZKDGfs/'),
                    'dtype': 'f8'},
              'xaxis': 'x2',
              'y': {'bdata': ('TuQyufIZAMClzZkVp0vzv9Pyssi7TO' ... 'OyyLtM7D8ezpkVp0vzP6d4s9z4HwFA'),
                    'dtype': 'f8'},
              'yaxis': 'y2'},
             {'line': {'color': 'gray', 'dash': 'dash'},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': 'rID9koMZ+7+sgP2Sgxn7Pw==', 'dtype': 'f8'},
              'xaxis': 'x2',
              'y': {'bdata': '4+X+Wsag/b/j5f5axqD9Pw==', 'dtype': 'f8'},
              'yaxis': 'y2'},
             {'marker': {'color': '#00CC96'},
              'nbinsx': 8,
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('AHCUXFOYlL8As7f1kwvevwCxt/WTC9' ... '55pFIS6b8AgMal/3+pv4D01Z5Squo/'),
                    'dtype': 'f8'},
              'xaxis': 'x3',
              'yaxis': 'y3'},
             {'marker': {'color': '#AB63FA', 'size': 8},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('3WHu+HMGUECR29IL1CtMQH7gBjSpPl' ... '9TKGCGUkDWqafNVLVSQI8O2nLZ7VJA'),
                    'dtype': 'f8'},
              'xaxis': 'x4',
              'y': {'bdata': ('JCskfr0HUED3Sr4z62dMQM0oEaCdIF' ... 'KczYS4UkCmYpzNhLhSQKZinM2EuFJA'),
                    'dtype': 'f8'},
              'yaxis': 'y4'},
             {'line': {'color': 'gray', 'dash': 'dash'},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': [56.342408636052944, 75.71639701170555],
              'xaxis': 'x4',
              'y': [56.342408636052944, 75.71639701170555],
              'yaxis': 'y4'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': '残差 vs 拟合值',
                                'x': 0.225,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': '正态概率图 (Q-Q)',
                                'x': 0.775,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': '残差直方图',
                                'x': 0.225,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 0.375,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                     

## 5. 响应面可视化

In [8]:
viz.surface_3d(0, 1, title="产率 vs 温度×pH (时间=中心点)")

/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: divide by zero encountered in matmul
  return X_design @ self.coefficients
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: overflow encountered in matmul
  return X_design @ self.coefficients
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: invalid value encountered in matmul
  return X_design @ self.coefficients


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'contours': {'z': {'highlightcolor': 'white', 'project': {'z': True}, 'show': True, 'usecolormap': True}},
              'opacity': 0.9,
              'type': 'surface',
              'x': {'bdata': ('AAAAAAAA+L/Byyl4OQX3v4OXU/ByCv' ... 'ByCvY/wMspeDkF9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAA+L/Byyl4OQX3v4OXU/ByCv' ... 'ByCvY/wMspeDkF9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'z': {'bdata': ('2SKIAZ4FSkDIdstgFKNKQKWe7rttOk' ... 'VGV4JNQJfHJ74zJ01AN4A5MfPFTEA='),
                    'dtype': 'f8',
                    'shape': '50, 50'}}],
    'layout': {'height': 600,
               'margin': {'b': 20, 'l': 20, 'r': 20, 't': 50},
               'scene': {'xaxis': {'title': {'text': '温度'}},
                         'yaxis': {'title': {'text': 'pH'}},
                         'zaxis': {'title': {'text': '响应值 (Y)'}}},
               'template': '...',
               'title': {'text': '产率 vs 温度×pH (时间=中心点)'},
               'width': 700}
})

In [9]:
viz.contour(0, 1, title="等高线: 温度×pH")

/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: divide by zero encountered in matmul
  return X_design @ self.coefficients
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: overflow encountered in matmul
  return X_design @ self.coefficients
/Users/james/Projects/self_projects/RSM/examples/../rsm/model.py:49: RuntimeWarning: invalid value encountered in matmul
  return X_design @ self.coefficients


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'colorbar': {'title': {'text': '响应值'}},
              'colorscale': [[0.0, 'rgb(49,54,149)'], [0.1, 'rgb(69,117,180)'],
                             [0.2, 'rgb(116,173,209)'], [0.3, 'rgb(171,217,233)'],
                             [0.4, 'rgb(224,243,248)'], [0.5, 'rgb(255,255,191)'],
                             [0.6, 'rgb(254,224,144)'], [0.7, 'rgb(253,174,97)'],
                             [0.8, 'rgb(244,109,67)'], [0.9, 'rgb(215,48,39)'],
                             [1.0, 'rgb(165,0,38)']],
              'contours': {'labelfont': {'size': 11}, 'showlabels': True},
              'type': 'contour',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'z': {'bdata': ('2SKIAZ4FSkCYRAL0AGhKQGEmFtEJyE' ... '0Prj1NQInH9qr9Ak1AN4A5MfPFTEA='),
                    'dtype': 'f8',
                    'shape': '80, 80'}}],
    'layout': {'height': 550,
               'template': '...',
               'title': {'text': '等高线: 温度×pH'},
               'width': 650,
               'xaxis': {'title': {'text': '温度'}},
               'yaxis': {'title': {'text': 'pH'}}}
})

In [10]:
viz.perturbation_plot()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': '#EF553B', 'width': 2.5},
              'mode': 'lines',
              'name': '温度',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L+EDz744IP3vwgffPDBB/' ... 'zwwQf3P4QPPvjgg/c/AAAAAAAA+D8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('QEBv1LFkS0CpQIR2+rxLQIONzbTDE0' ... 'kaqM1QQAZYvAMqsVBA9XnJOuyTUEA='),
                    'dtype': 'f8'}},
             {'line': {'color': '#636EFA', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L+EDz744IP3vwgffPDBB/' ... 'zwwQf3P4QPPvjgg/c/AAAAAAAA+D8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('8g3mjcgbUUBK7kQDGzBRQCQBp/XuQ1' ... '9hMaRPQHtZKQn3bE9Al8VZqr80T0A='),
                    'dtype': 'f8'}},
             {'line': {'color': '#00CC96', 'width': 2.5},
              'mode': 'lines',
              'name': '时间',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L+EDz744IP3vwgffPDBB/' ... 'zwwQf3P4QPPvjgg/c/AAAAAAAA+D8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('5fQ6IigIUEClM4C4BCBQQOQ7PjZ4N1' ... 'j3TptRQFb8UUJZi1FAb13UdPp6UUA='),
                    'dtype': 'f8'}}],
    'layout': {'height': 450,
               'legend': {'x': 0.01, 'y': 0.99},
               'template': '...',
               'title': {'text': '扰动图 (Perturbation Plot)'},
               'width': 650,
               'xaxis': {'title': {'text': '编码值'}},
               'yaxis': {'title': {'text': '响应值 (Y)'}}}
})

In [11]:
viz.interaction_plot(0, 1)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': '#636EFA', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH=-1.5',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('2SKIAZ4FSkCYRAL0AGhKQGEmFtEJyE' ... 'nAu0xA3xlO3AppTECAvqBJ+xNMQA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#EF553B', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH=-0.8',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('ryYFOeDjS0ByTbtxRExMQD40C5VOsk' ... 'gyNFBAWlIKPdgNUEBXTqMtosxPQA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#00CC96', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH=0.0',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('QEBv1LFkS0AGbGFTF9NLQNVX7bwiP0' ... 'bM21BAoiLkvXK4UED1eck67JNQQA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#AB63FA', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH=0.8',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('im/G0xKISEBToPSYefxIQCWRvEiGbk' ... 'WtVFBAxn208FQ0UECb17cQzxJQQA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#FFA15A', 'width': 2.5},
              'mode': 'lines',
              'name': 'pH=1.5',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAA+L/JnoGodGT3v5I9A1HpyP' ... 'HpyPY/yp6BqHRk9z8AAAAAAAD4Pw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('irQKNwNOQ0BW6nRCa8hDQCzgeDh5QE' ... '+uPU1Aisf2qv0CTUA3gDkx88VMQA=='),
                    'dtype': 'f8'}}],
    'layout': {'height': 450,
               'template': '...',
               'title': {'text': '交互作用图: 温度 × pH'},
               'width': 650,
               'xaxis': {'title': {'text': '温度'}},
               'yaxis': {'title': {'text': '响应值 (Y)'}}}
})

## 6. 优化分析

In [12]:
optimizer = RSMOptimizer(model, factors)
result = optimizer.optimize(maximize=True)

print("最优条件:")
for name, coded, natural in zip(result.factor_names, result.optimal_coded, result.optimal_natural):
    print(f"  {name}: {natural:.2f} (编码值: {coded:.4f})")
print(f"  预测最大产率: {result.predicted_response:.2f}%")

最优条件:
  温度: 42.68 (编码值: 0.2678)
  pH: 6.66 (编码值: -0.1683)
  时间: 44.73 (编码值: 0.2365)
  预测最大产率: 75.79%


In [13]:
canonical = optimizer.canonical_analysis()
print(f"曲面形态: {canonical['曲面形态']}")
print(f"驻点预测值: {canonical['驻点预测值']:.4f}")
print(f"特征值: {canonical['特征值']}")

曲面形态: 极大值 (Maximum)
驻点预测值: 75.7948
特征值: [-6.70003021 -3.99761694 -3.3702334 ]
